#Question and Answering System

##1. Environment Setup & Data Collection

In [6]:
def collect_data():
    """
    Loads an optimized, large subset of the SQuAD dataset (20k train, 2k validation)
    to balance high evaluation metrics with fast training times on a free T4 GPU.
    """
    !pip install -q transformers datasets evaluate accelerate

    from datasets import load_dataset
    from transformers import AutoTokenizer

    print("Loading optimized SQuAD dataset slice from Hugging Face...")
    # 20k train / 2k validation provides high accuracy without multi-hour training bottlenecks
    dataset = load_dataset("rajpurkar/squad", split={"train": "train[:20000]", "validation": "validation[:2000]"})

    # Upgraded model checkpoint with superior semantic comprehension
    model_checkpoint = "deepset/roberta-base-squad2"
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

    print("Optimized data collection completed successfully!")
    return dataset, tokenizer

# Execute collection
dataset, tokenizer = collect_data()

Loading optimized SQuAD dataset slice from Hugging Face...
Optimized data collection completed successfully!


##2. Data Preparation & Preprocessing

In [7]:
def prepare_and_preprocess_data(dataset, tokenizer):
    """
    Tokenizes text passages and aligns SQuAD question-answer character spans
    with token indices so the model knows where answers start and end.
    """
    max_length = 384    # Maximum token length for input chunks
    doc_stride = 128    # Overlap stride between chunks if context is too long

    def preprocess_function(examples):
        # Strip trailing/leading whitespaces from questions
        questions = [q.strip() for q in examples["question"]]

        # Tokenize questions and contexts with truncation and sliding window overflow
        inputs = tokenizer(
            questions,
            examples["context"],
            max_length=max_length,
            stride=doc_stride,
            return_overflowing_tokens=True,
            return_offsets_mapping=True,
            padding="max_length",
            truncation="only_second", # Truncate context if it exceeds max_length, not the question
        )

        # Map overflowing token chunks back to original sample indices
        sample_map = inputs.pop("overflow_to_sample_mapping")
        offset_mapping = inputs.pop("offset_mapping")
        examples_answers = examples["answers"]

        start_positions = []
        end_positions = []

        for i, offsets in enumerate(offset_mapping):
            sample_idx = sample_map[i]
            answer = examples_answers[sample_idx]

            # Get character-level start and end positions of the ground truth answer
            start_char = answer["answer_start"][0]
            end_char = start_char + len(answer["text"][0])
            sequence_ids = inputs.sequence_ids(i)

            # Locate the start token index of the context within the input sequence
            token_start_index = 0
            while sequence_ids[token_start_index] != 1:
                token_start_index += 1

            # Locate the end token index of the context within the input sequence
            token_end_index = len(sequence_ids) - 1
            while sequence_ids[token_end_index] != 1:
                token_end_index -= 1

            # Check if the answer falls completely outside the current token span window
            if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
                start_positions.append(0)
                end_positions.append(0)
            else:
                # Find the token index corresponding to the answer's start character
                current_idx = token_start_index
                while current_idx <= token_end_index and offsets[current_idx][0] <= start_char:
                    current_idx += 1
                start_positions.append(current_idx - 1)

                # Find the token index corresponding to the answer's end character
                current_idx = token_end_index
                while current_idx >= token_start_index and offsets[current_idx][1] >= end_char:
                    current_idx -= 1
                end_positions.append(current_idx + 1)

        # Append computed token positions into the inputs dictionary
        inputs["start_positions"] = start_positions
        inputs["end_positions"] = end_positions
        return inputs

    # Map preprocessing across the entire dataset structure
    tokenized_dataset = dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=dataset["train"].column_names,
    )
    print("Data preparation and preprocessing completed!")
    return tokenized_dataset

# Execute preparation and preprocessing
tokenized_dataset = prepare_and_preprocess_data(dataset, tokenizer)

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Data preparation and preprocessing completed!


## 3. Model Building

In [9]:
def build_model():
    """
    Initializes the high-performance RoBERTa architecture fine-tuned for Question Answering.
    """
    from transformers import AutoModelForQuestionAnswering
    model_checkpoint = "deepset/roberta-base-squad2"

    # Load RoBERTa QA model
    model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)
    print("High-performance RoBERTa model built successfully!")
    return model

# Execute model building
model = build_model()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

High-performance RoBERTa model built successfully!


##4. Model Training

In [10]:
def train_model(model, tokenized_dataset, tokenizer):
    """
    Fine-tunes RoBERTa with optimized parameters (3 epochs, gradient accumulation)
    for fast execution and high accuracy on the T4 GPU.
    """
    from transformers import Trainer, TrainingArguments, DefaultDataCollator

    data_collator = DefaultDataCollator()

    # Fast, memory-efficient training arguments optimized for T4 GPU
    training_args = TrainingArguments(
        output_dir="./results_roberta",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,                  # Stable learning rate for RoBERTa convergence
        per_device_train_batch_size=8,       # Safe batch size for RoBERTa in T4 VRAM
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=2,       # Effective batch size = 16 (speeds up optimization)
        num_train_epochs=3,                  # 3 epochs for fast, high-quality learning
        weight_decay=0.01,
        fp16=True,                           # Mixed precision for rapid T4 GPU execution
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        data_collator=data_collator,
    )

    print("Starting fast-optimized RoBERTa training on T4 GPU...")
    trainer.train()
    print("Training completed successfully!")
    return trainer

# Execute training
trainer = train_model(model, tokenized_dataset, tokenizer)

Starting fast-optimized RoBERTa training on T4 GPU...


Epoch,Training Loss,Validation Loss
1,0.638813,0.833546
2,0.420900,0.949863
3,0.281428,1.065859


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training completed successfully!


## 5. Model Testing & Inference Pipeline

In [ ]:
def test_model_generalization(model, tokenizer):
    """
    Tests the fine-tuned model's inference capability on both standard questions
    and a completely outside/unseen real-world paragraph, ensuring device compatibility.
    """
    import torch

    outside_context = "Artificial intelligence engineering teams at TechCorp deployed a new LLM inference cluster in Frankfurt on October 12, 2026, reducing latency by 45 percent."
    outside_question = "When was the LLM inference cluster deployed?"

    # Automatically detect the model's device (GPU/CPU) and move inputs there
    device = next(model.parameters()).device
    inputs = tokenizer(outside_question, outside_context, return_tensors="pt").to(device)

    model.eval() # Switch model to evaluation mode
    with torch.no_grad():
        outputs = model(**inputs)

    # Extract the highest scoring start and end token indices
    answer_start_index = torch.argmax(outputs.start_logits)
    answer_end_index = torch.argmax(outputs.end_logits)

    # Decode token IDs back into the predicted text string
    predict_answer_tokens = inputs.input_ids[0, answer_start_index : answer_end_index + 1]
    answer = tokenizer.decode(predict_answer_tokens)

    print("=== Outside Text Generalization Test ===")
    print(f"Context: {outside_context}")
    print(f"Question: {outside_question}")
    print(f"Model Extracted Answer: {answer}\n")

# Execute testing and inference
test_model_generalization(model, tokenizer)

## 6.Model Evaluation (Exact Match & Token-Level F1 Word Generalization)

In [ ]:
def evaluate_model_with_generalization(model, tokenizer, val_dataset):
    """
    Evaluates model performance across validation records using Exact Match (EM)
    and a Token-Level F1 score with proper GPU tensor placement and index conversion.
    """
    import torch
    import collections

    print("Running evaluation on validation dataset...")
    model.eval()

    exact_matches = 0
    f1_scores = []
    total = min(100, len(val_dataset)) # Evaluates the first 100 validation samples

    # Detect model device (e.g., cuda:0)
    device = next(model.parameters()).device

    def normalize_text(text):
        """Removes punctuation and capitalization for fair text comparison."""
        return "".join(ch for ch in text.lower() if ch.isalnum() or ch.isspace())

    def compute_f1(prediction, ground_truth):
        """Calculates token overlap F1 score to handle partial word matches and generalization."""
        pred_toks = normalize_text(prediction).split()
        truth_toks = normalize_text(ground_truth).split()

        if len(pred_toks) == 0 or len(truth_toks) == 0:
            return int(pred_toks == truth_toks)

        common = collections.Counter(pred_toks) & collections.Counter(truth_toks)
        num_same = sum(common.values())

        if num_same == 0:
            return 0

        precision = 1.0 * num_same / len(pred_toks)
        recall = 1.0 * num_same / len(truth_toks)
        f1 = (2 * precision * recall) / (precision + recall)
        return f1

    raw_val = dataset["validation"]

    for i in range(total):
        context = raw_val[i]["context"]
        question = raw_val[i]["question"]
        true_answers = raw_val[i]["answers"]["text"]

        # Tokenize and push tensors to the GPU device
        inputs = tokenizer(question, context, return_tensors="pt", truncation=True, max_length=384).to(device)
        with torch.no_grad():
            outputs = model(**inputs)

        # Get predicted span indices and convert them to standard integers using .item()
        start_idx = torch.argmax(outputs.start_logits).item()
        end_idx = torch.argmax(outputs.end_logits).item()

        # Decode predicted answer tokens back to text
        pred_ans = tokenizer.decode(inputs.input_ids[0, start_idx:end_idx + 1])

        # Compute Exact Match metric across valid answers
        em = max(1 if normalize_text(pred_ans) == normalize_text(t) else 0 for t in true_answers)
        exact_matches += em

        # Compute Token F1 word generalization metric
        f1 = max(compute_f1(pred_ans, t) for t in true_answers)
        f1_scores.append(f1)

    # Calculate final average percentages
    avg_em = (exact_matches / total) * 100
    avg_f1 = (sum(f1_scores) / total) * 100

    print("=== Evaluation Results with Word Generalization ===")
    print(f"Exact Match (EM) Score: {avg_em:.2f}%")
    print(f"Token-Level F1 Generalization Score: {avg_f1:.2f}%")

# Execute evaluation
evaluate_model_with_generalization(model, tokenizer, tokenized_dataset["validation"])

In [ ]:
def evaluate_precision_recall_f1(model, tokenizer, val_dataset):
    """
    Computes corpus-wide or macro-averaged Precision, Recall, and F1
    for the fine-tuned QA model.
    """
    import torch
    import collections

    print("Calculating Precision, Recall, and F1 metrics...")
    model.eval()

    precisions = []
    recalls = []
    f1_scores = []
    total = min(100, len(val_dataset))

    device = next(model.parameters()).device

    def normalize_text(text):
        return "".join(ch for ch in text.lower() if ch.isalnum() or ch.isspace())

    raw_val = dataset["validation"]

    for i in range(total):
        context = raw_val[i]["context"]
        question = raw_val[i]["question"]
        true_answers = raw_val[i]["answers"]["text"]

        inputs = tokenizer(question, context, return_tensors="pt", truncation=True, max_length=384).to(device)
        with torch.no_grad():
            outputs = model(**inputs)

        start_idx = torch.argmax(outputs.start_logits).item()
        end_idx = torch.argmax(outputs.end_logits).item()
        pred_ans = tokenizer.decode(inputs.input_ids[0, start_idx:end_idx + 1])

        # Evaluate against all valid ground-truth answers for this question
        best_p, best_r, best_f1 = 0, 0, 0

        for t in true_answers:
            pred_toks = normalize_text(pred_ans).split()
            truth_toks = normalize_text(t).split()

            if len(pred_toks) == 0 or len(truth_toks) == 0:
                score = 1.0 if pred_toks == truth_toks else 0.0
                precisions.append(score)
                recalls.append(score)
                f1_scores.append(score)
                continue

            common = collections.Counter(pred_toks) & collections.Counter(truth_toks)
            num_same = sum(common.values())

            if num_same > 0:
                p = num_same / len(pred_toks)
                r = num_same / len(truth_toks)
                f1 = (2 * p * r) / (p + r)
            else:
                p, r, f1 = 0.0, 0.0, 0.0

            if f1 > best_f1:
                best_p, best_r, best_f1 = p, r, f1

        precisions.append(best_p)
        recalls.append(best_r)
        f1_scores.append(best_f1)

    avg_precision = (sum(precisions) / total) * 100
    avg_recall = (sum(recalls) / total) * 100
    avg_f1 = (sum(f1_scores) / total) * 100

    print("=== Detailed Evaluation Metrics ===")
    print(f"Average Precision: {avg_precision:.2f}%")
    print(f"Average Recall:    {avg_recall:.2f}%")
    print(f"Overall F1 Score:  {avg_f1:.2f}%")

# Execute detailed evaluation
evaluate_precision_recall_f1(model, tokenizer, tokenized_dataset["validation"])

In [11]:
def deploy_model(model, tokenizer):
    """
    Serializes and saves the fine-tuned model weights and tokenizer directly
    to the connected Google Drive folder for downstream API integration or production deployment.
    """
    # Updated path pointing directly to your Google Drive directory
    save_directory = "/content/drive/MyDrive/Data Science (FULL)/NLP/Project -13/saved_qa_model"

    # Save the model weights and tokenizer
    model.save_pretrained(save_directory)
    tokenizer.save_pretrained(save_directory)

    print(f"Model successfully saved to {save_directory}. Ready for downstream deployment!")

# Execute deployment preparation
deploy_model(model, tokenizer)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model successfully saved to /content/drive/MyDrive/Data Science (FULL)/NLP/Project -13/saved_qa_model. Ready for downstream deployment!
